#### imports et chargement 

In [ ]:
from sklearn.experimental import enable_iterative_imputer   
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

quantitative = {
    "continuous": ["Sleep_Hours", "Study_Hours_Per_Week", "GPA", "Social_Media_Hours",
                   "Screen_Time_Hours", "Financial_Stress", "Family_Support", "Attendance_Rate"],
    "discrete": ["Age", "Year_of_Study"],
}
qualitative = {
    "nominal": ["Gender", "Part_Time_Job", "Previous_Mental_Health_History"],
    "ordinal": ["Physical_Activity", "Diet_Quality"],
}

###  split le dataset 

In [4]:
X = data.drop(columns=["Student_ID",cible])
y=data[cible]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

### Traiter les valuers manquants

In [16]:
manquants = (data.isna().sum()).round(2).sort_values(ascending=False)
manquants[manquants > 0]

Financial_Stress      91
Diet_Quality          69
Sleep_Hours           62
Social_Media_Hours    58
Family_Support        54
Attendance_Rate       47
GPA                   38
dtype: int64

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler)

from src.config import (NUMERIC_COLS, ENGINEERED_COLS, ORDINAL_COLS, NOMINAL_COLS)


def add_features(df):
    df = df.copy()
    df["Sleep_Deficit"] = (7 - df["Sleep_Hours"]).clip(lower=0)
    df["Non_Social_Screen_Time"] = (df["Screen_Time_Hours"] - df["Social_Media_Hours"]).clip(lower=0)
    df["Stress_Support_Gap"] = df["Financial_Stress"] - df["Family_Support"]
    return df


class IQRClipper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.percentile(X, [25, 75], axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


def build_preprocessor():
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clipper", IQRClipper(factor=1.5)),
        ("scaler", StandardScaler()),
    ])
    ordinal = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            categories=list(ORDINAL_COLS.values()),
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )),
    ])
    nominal = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, NUMERIC_COLS + ENGINEERED_COLS),
        ("ord", ordinal, list(ORDINAL_COLS)),
        ("nom", nominal, NOMINAL_COLS),
    ])


def build_feature_prep():
    return Pipeline([
        ("features", FunctionTransformer(add_features)),
        ("preprocessor", build_preprocessor()),
    ])


def build_pipeline(model):
    return Pipeline([
        ("prep", build_feature_prep()),
        ("model", model),
    ])
    